# Dilbert

Dilbert is the interactive Zenoh tool for Nachtlicht. Execute the cells from top to bottom. The notebook opens a direct Zenoh session to the Windows router, requests colour changes and displays the ESP32 events.

## 1. Load the local router configuration

Copy `config.example.py` to `config.py` and set the router locator before running this cell. `config.py` is intentionally not committed.

In [1]:
from config import ROUTER_LOCATOR

COLOR_NEXT_KEYEXPR = "nachtlicht/led/color/next"
COLOR_CHANGED_KEYEXPR = "nachtlicht/led/color/changed"
BRIGHTNESS_CHANGED_KEYEXPR = "nachtlicht/led/brightness/changed"
COLOR_NEXT_PAYLOAD = "next"

print(f"Router: {ROUTER_LOCATOR}")
print(f"Command key: {COLOR_NEXT_KEYEXPR}")
print(f"Event keys: {COLOR_CHANGED_KEYEXPR}, {BRIGHTNESS_CHANGED_KEYEXPR}")

Router: tcp/10.0.0.63:7447
Command key: nachtlicht/led/color/next
Event keys: nachtlicht/led/color/changed, nachtlicht/led/brightness/changed


## 2. Open a Zenoh session

Multicast scouting is disabled deliberately: Dilbert connects only to the configured Windows router.

In [2]:
import json

import zenoh

config = zenoh.Config()
config.insert_json5("mode", json.dumps("client"))
config.insert_json5("connect/endpoints", json.dumps([ROUTER_LOCATOR]))
config.insert_json5("scouting/multicast/enabled", "false")

session = zenoh.open(config)
print("Zenoh session opened.")

Zenoh session opened.


## 3. Declare the publisher and event subscribers

The publisher sends colour-change requests. The subscribers receive the ESP32 colour and brightness events through thread-safe Zenoh queues; the GUI consumes those queues in the Jupyter event loop.

In [3]:
publisher = session.declare_publisher(COLOR_NEXT_KEYEXPR)
color_changed_subscriber = session.declare_subscriber(COLOR_CHANGED_KEYEXPR)
brightness_changed_subscriber = session.declare_subscriber(BRIGHTNESS_CHANGED_KEYEXPR)
print(f"Publisher declared for {COLOR_NEXT_KEYEXPR}.")
print("Event subscribers declared.")

Publisher declared for nachtlicht/led/color/next.
Event subscribers declared.


## 4. Control and observe Nachtlicht

Click the button to request the next colour. The asynchronous task polls the Zenoh subscriber queues without blocking the notebook, then updates the widgets from the Jupyter event loop. The brightness-event counter is the live heartbeat.

In [4]:
import asyncio
import json

import ipywidgets as widgets
from IPython.display import display

color_events = 0
brightness_events = 0

next_color_button = widgets.Button(description="Next colour", button_style="primary")
color_status = widgets.HTML(value="Last colour: waiting for an event")
brightness_status = widgets.HTML(value="Brightness heartbeat: waiting for an event")

def request_next_colour(_button):
    publisher.put(COLOR_NEXT_PAYLOAD)

def update_colour(sample):
    global color_events
    event = json.loads(sample.payload.to_string())
    color_events += 1
    color_status.value = f"Last colour: {event['color']} (event {event['sequence']}, received {color_events})"

def update_brightness(sample):
    global brightness_events
    event = json.loads(sample.payload.to_string())
    brightness_events += 1
    brightness_status.value = f"Brightness heartbeat: {event['brightness']} (event {event['sequence']}, received {brightness_events})"

async def receive_events():
    while True:
        while sample := color_changed_subscriber.try_recv():
            update_colour(sample)
        while sample := brightness_changed_subscriber.try_recv():
            update_brightness(sample)
        await asyncio.sleep(0.1)

next_color_button.on_click(request_next_colour)
event_task = asyncio.create_task(receive_events())
display(widgets.VBox([next_color_button, color_status, brightness_status]))

## 5. Close the session

Run this cell before restarting the kernel or shutting down JupyterLab. It stops the event task before closing the Zenoh resources.

In [ ]:
event_task.cancel()
color_changed_subscriber.undeclare()
brightness_changed_subscriber.undeclare()
publisher.undeclare()
session.close()
print("Zenoh session closed.")